# GR1T1 OSC Pose Controller Sweep
Run a wandb hyperparameter sweep on the GR1T1 `Lift` task to minimize end-effector pose error with the OSC pose controller utilities included in this repo.



## Notebook Overview
- Push your updated `robosuite` fork to GitHub, then set `REPO_URL` and `BRANCH` below.
- This notebook installs the repo in editable mode on Colab, sets up MuJoCo for headless rendering, and logs to wandb.
- Sweeps default to the pose targets stored in `osc_evals/ee_targets_2.json` and skip video uploads for faster runtime.
- Update the wandb project/entity to match your workspace before running.



In [ ]:
# Install the desired Python version if not already present
!sudo apt-get update
!sudo apt-get install python3.10 python3.10-dev # Example for Python 3.8

# Configure alternatives
!sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.8 1
!sudo update-alternatives --config python3

# Verify the change
!python3 --version

Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [343 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,827 kB]
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [6,181 k

In [ ]:
!sudo apt update
!sudo apt install python3-pip

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
49 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as re

In [4]:
import os

REPO_URL = "https://github.com/Sarthak-Dayal/robosuite.git"  # TODO: update to your fork
BRANCH = "osc-evals-framework"
PROJECT = "osc-controller-evals"
ENTITY = "Sarthak-Dayal"  # e.g., "my-team"; set to None for personal wandb accounts
ROBOT = "GR1T1RightArmOnly"

os.environ.setdefault("MUJOCO_GL", "egl")

print(f"Repo: {REPO_URL}")
print(f"Branch: {BRANCH}")
print(f"wandb project: {PROJECT}")
print(f"wandb entity: {ENTITY}")
print(f"Robot: {ROBOT}")



Repo: https://github.com/Sarthak-Dayal/robosuite.git
Branch: osc-evals-framework
wandb project: osc-controller-evals
wandb entity: Sarthak-Dayal
Robot: GR1T1RightArmOnly


In [7]:
import os
import pathlib
import subprocess
import sys

repo_name = REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")
if not pathlib.Path(repo_name).exists():
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL], check=True)
os.chdir(repo_name)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wandb", "mujoco"], check=True)



CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'wandb', 'mujoco'], returncode=0)

In [ ]:
%pip install -r /content/robosuite/requirements.txt

Obtaining file:///content/robosuite/robosuite/robosuite (from -r /content/robosuite/requirements.txt (line 1))
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 38.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 132.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.1/92.1 KB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 35.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.7/91.7 KB 10.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.7/37.7 MB 63.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 896.8/896.8 KB 62.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 373.7/373.7 KB 39.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import wandb
wandb.login()



/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: sarthakdayal (sarthak-dayal) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [79]:
import json

import sys

clean = []
for p in sys.path:
    # Keep default system paths only
    if not ("robosuite" in p):
        clean.append(p)

clean.append('content/robosuite')
clean.append('content/robosuite/robosuite')
sys.path = clean
print(sys.path)


from robosuite.osc_evals.empirical.gr1t1_sweep import default_search_space

sweep_config = default_search_space(metric="overall_position_error")
sweep_config["name"] = "gr1t1_pose_controller"
print(json.dumps(sweep_config, indent=2))



[robosuite WARNING] Could not import robosuite_models. Some robots may not be available. If you want to use these robots, please install robosuite_models from source (https://github.com/ARISE-Initiative/robosuite_models) or through pip install. (__init__.py:30)
[robosuite WARNING] Could not import robosuite_models. Some robots may not be available. If you want to use these robots, please install robosuite_models from source (https://github.com/ARISE-Initiative/robosuite_models) or through pip install. (__init__.py:30)
[robosuite WARNING] Could not load the mink-based whole-body IK. Make sure you install related import properly, otherwise you will not be able to use the default IK controller setting for GR1 robot. (__init__.py:40)
[robosuite WARNING] Could not load the mink-based whole-body IK. Make sure you install related import properly, otherwise you will not be able to use the default IK controller setting for GR1 robot. (__init__.py:40)


['/content', '/env/python', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/usr/local/lib/python3.12/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.12/dist-packages/IPython/extensions', '/root/.ipython', '/content', '/content', '/content', '/content', '/content', '/content', 'content/robosuite', 'content/robosuite/robosuite']
{
  "method": "bayes",
  "metric": {
    "name": "overall_position_error",
    "goal": "minimize"
  },
  "parameters": {
    "kp": {
      "min": 80.0,
      "max": 250.0
    },
    "damping_ratio": {
      "min": 0.4,
      "max": 2.0
    },
    "pos_output_max": {
      "min": 0.03,
      "max": 0.07
    },
    "ori_output_max": {
      "values": [
        0.3,
        0.4,
        0.5,
        0.6
      ]
    },
    "hold_duration": {
      "values": [
        100,
        125,
        150
      ]
    }
  },
  "name": "gr1t1_pose_controller"
}


In [81]:
import wandb
sweep_id = wandb.sweep(sweep=sweep_config, project=PROJECT, entity=ENTITY)
print("Sweep ID:", sweep_id)



/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


Create sweep with ID: nv9btd8r
Sweep URL: https://wandb.ai/sarthak-dayal/osc-controller-evals/sweeps/nv9btd8r
Sweep ID: nv9btd8r


In [83]:
from robosuite.osc_evals.empirical.gr1t1_sweep import run_gr1t1_pose_trial


def sweep_runner():
    with wandb.init(project=PROJECT, entity=ENTITY) as run:
        trial_config = dict(run.config)
        results = run_gr1t1_pose_trial(
            trial_config,
            wandb_project=PROJECT,
            wandb_entity=ENTITY,
            log_videos=False,
            robot_name=ROBOT,
        )
        if results:
            numeric_results = {k: v for k, v in results.items() if isinstance(v, (int, float))}
            if numeric_results:
                wandb.log(numeric_results)
        print("Completed run with results:", results)



In [90]:
run_gr1t1_pose_trial(
    {},
    wandb_project=PROJECT,
    wandb_entity=ENTITY,
    log_videos=False,
    robot_name=ROBOT,
)

ModuleNotFoundError: No module named 'robosuite.controllers'

In [84]:
SWEEP_TRIALS = 1000
wandb.agent(sweep_id, function=sweep_runner, count=SWEEP_TRIALS)



wandb: Agent Starting Run: kmcfzusj with config:
wandb: 	damping_ratio: 1.507003109195511
wandb: 	hold_duration: 150
wandb: 	kp: 135.42189746791172
wandb: 	ori_output_max: 0.4
wandb: 	pos_output_max: 0.05327430831732719
wandb: Currently logged in as: sarthakdayal (sarthak-dayal) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Traceback (most recent call last):
  File "/tmp/ipython-input-925026676.py", line 7, in sweep_runner
    results = run_gr1t1_pose_trial(
              ^^^^^^^^^^^^^^^^^^^^^
  File "/content/robosuite/osc_evals/empirical/gr1t1_sweep.py", line 100, in run_gr1t1_pose_trial
    results = demo_runner.run_demo(
              ^^^^^^^^^^^^^^^^^^^^^
  File "/content/robosuite/osc_evals/runner.py", line 59, in run_demo
    self.current_env = self.env_manager.create_environment(
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/robosuite/osc_evals/environment.py", line 48, in create_environment
    arm_controller_config = suite.load_part_controller_config(
                            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/robosuite/robosuite/controllers/parts/controller_factory.py", line 45, in load_part_controller_config
    from robosuite.controllers import ALL_PART_CONTROLLERS
ModuleNotFoundError: No module named 'robosuite.controllers'


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/wandb/agents/pyagent.py", line 297, in _run_job
    self._function()
  File "/tmp/ipython-input-925026676.py", line 7, in sweep_runner
    results = run_gr1t1_pose_trial(
              ^^^^^^^^^^^^^^^^^^^^^
  File "/content/robosuite/osc_evals/empirical/gr1t1_sweep.py", line 100, in run_gr1t1_pose_trial
    results = demo_runner.run_demo(
              ^^^^^^^^^^^^^^^^^^^^^
  File "/content/robosuite/osc_evals/runner.py", line 59, in run_demo
    self.current_env = self.env_manager.create_environment(
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/robosuite/osc_evals/environment.py", line 48, in create_environment
    arm_controller_config = suite.load_part_controller_config(
                            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/robosuite/robosuite/controllers/parts/controller_factory.py", line 45, in load_part_controller_config
    from robosuite.c

Traceback (most recent call last):
  File "/tmp/ipython-input-925026676.py", line 7, in sweep_runner
    results = run_gr1t1_pose_trial(
              ^^^^^^^^^^^^^^^^^^^^^
  File "/content/robosuite/osc_evals/empirical/gr1t1_sweep.py", line 100, in run_gr1t1_pose_trial
    results = demo_runner.run_demo(
              ^^^^^^^^^^^^^^^^^^^^^
  File "/content/robosuite/osc_evals/runner.py", line 59, in run_demo
    self.current_env = self.env_manager.create_environment(
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/robosuite/osc_evals/environment.py", line 48, in create_environment
    arm_controller_config = suite.load_part_controller_config(
                            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/robosuite/robosuite/controllers/parts/controller_factory.py", line 45, in load_part_controller_config
    from robosuite.controllers import ALL_PART_CONTROLLERS
ModuleNotFoundError: No module named 'robosuite.controllers'


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/wandb/agents/pyagent.py", line 297, in _run_job
    self._function()
  File "/tmp/ipython-input-925026676.py", line 7, in sweep_runner
    results = run_gr1t1_pose_trial(
              ^^^^^^^^^^^^^^^^^^^^^
  File "/content/robosuite/osc_evals/empirical/gr1t1_sweep.py", line 100, in run_gr1t1_pose_trial
    results = demo_runner.run_demo(
              ^^^^^^^^^^^^^^^^^^^^^
  File "/content/robosuite/osc_evals/runner.py", line 59, in run_demo
    self.current_env = self.env_manager.create_environment(
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/robosuite/osc_evals/environment.py", line 48, in create_environment
    arm_controller_config = suite.load_part_controller_config(
                            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/robosuite/robosuite/controllers/parts/controller_factory.py", line 45, in load_part_controller_config
    from robosuite.c

wandb: Ctrl + C detected. Stopping sweep.


### Notes
- Each agent run reuses the shared wandb run created inside `sweep_runner`, so the empirical test will not re-initialize wandb and video logging stays off by default.
- The helper defaults to `GR1T1RightArmOnly` to match the single-arm controller wiring; override `ROBOT` if you add support for other variants.
- To evaluate a single configuration outside the sweep, call `run_gr1t1_pose_trial({...})` directly (optionally set `log_videos=True` once you have promising candidates).
- Review the `overall_position_error` metric in the wandb dashboard to pick the best controller gains, then re-run the helper with that config to record higher-quality videos if needed.

